In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
# load the data
train_df = pd.read_csv('data/train_synthetic.csv')
test_df = pd.read_csv('data/test_synthetic.csv')
greeks_df = pd.read_csv('data/greeks_synthetic.csv')

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
train_df = pd.merge(train_df, greeks_df, on="Id")

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
# Remove the first column
train_df = train_df.drop("Id", axis=1)
test_df = test_df.drop("Id", axis=1)

In [5]:
# --- [CELL 4]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
# === BEFORE (original) ===
# # One-hot encoding
# encoder = OneHotEncoder(handle_unknown="ignore")
# train_df = pd.get_dummies(train_df, columns=list(train_df))
# test_df = pd.get_dummies(test_df, columns=list(test_df))

# === AFTER (edited) ===
# Identify categorical columns that exist in both train and test
cat_cols_train = train_df.select_dtypes(include=['object']).columns.tolist()
cat_cols_test = test_df.select_dtypes(include=['object']).columns.tolist()
cat_cols = list(set(cat_cols_train) & set(cat_cols_test))
print("Categorical columns to encode (both train and test):", cat_cols)

# Identify categorical columns only in train (likely extra labels to remove)
extra_cat_cols = list(set(cat_cols_train) - set(cat_cols_test))
if extra_cat_cols:
    print("Dropping extra categorical columns from train:", extra_cat_cols)
    train_df = train_df.drop(columns=extra_cat_cols)

# Apply one-hot encoding only to common categorical columns
train_df = pd.get_dummies(train_df, columns=cat_cols, drop_first=True)
test_df = pd.get_dummies(test_df, columns=cat_cols, drop_first=True)

# Align columns between train and test (ensure same feature space)
# Add missing columns to test as zeros, and remove extras
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

Categorical columns to encode (both train and test): ['EJ']
Dropping extra categorical columns from train: ['Gamma', 'Epsilon', 'Alpha', 'Beta', 'Delta']
Train shape: (624, 57)
Test shape: (5, 57)


In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
# Data processing
scaler = StandardScaler()
train_df = scaler.fit_transform(train_df)
test_df = scaler.transform(test_df)

In [7]:
# Validate the scaler transform fix: train/test must share identical fitted feature space
assert hasattr(scaler, "n_features_in_"), "Scaler should be fitted on training features"
assert train_df.shape[1] == test_df.shape[1], "Train/test feature counts must match after encoding and alignment"
assert train_df.shape[1] == scaler.n_features_in_, "Scaled train feature count must match scaler fit schema"
assert test_df.shape[1] == scaler.n_features_in_, "Scaled test feature count must match scaler fit schema"